# 1 — The Cartesian oval, and what ray theory predicts on it

The surface that images a point A onto a point A′ with no aberration is the
Cartesian oval: the locus where the two-leg optical path
$n_1 d_1 + n_2 d_2$ is constant.  Everything in this notebook is geometry and
ray optics; no field is propagated yet.

What gets checked here, in order:

1. the profile really is stigmatic — measured from the coordinates, not
   assumed by the parametrisation;
2. Snell's law holds at every point;
3. the ray-tube identity relating the two solid angles;
4. the two independent ways of writing the amplitude on the exit reference
   sphere agree — $P = t_s\,d_2/d_1$ and $P=\sqrt{T_s (n_1/n_2)\,w_1/w_2}$;
5. what that pupil looks like, against the hypothesis of constant amplitude.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, matplotlib.pyplot as plt
from nbstyle import *

from diffractor.optics import Medium, stigmatic_interface
from diffractor.analysis import (two_leg_opl, invariant_opl, opd_waves,
                                 predicted_pupil, energy_through_sphere)
from diffractor.propagation.transport import (ray_tube_amplitude,
                                              stigmatic_pupil,
                                              stigmatic_pupil_from_tubes)
from diffractor.scattering import t_s, T_s

LAM = 1.0
n1, n2 = 1.0, 1.5
zo, zi = -16.0, 8.0

itf = stigmatic_interface(Medium(n1), Medium(n2), zo, zi)
g = itf.sample(20001, i1_max_deg=80.0)
NA = n2 * g["sin2"][-1]
print(f"rim at r = {g['r'][-1]:.3f} λ, z = {g['z'][-1]:.3f} λ")
print(f"NA = n2 sin θ2,max = {NA:.4f}   (θ2,max = {np.degrees(g['th2'][-1]):.2f}°)")
print(f"stigmatic constant C = n1|zo| + n2 zi = {invariant_opl(n1, n2, zo, zi):.3f} λ")

## The profile

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.plot(g["z"], g["r"], color=BLUE, lw=2.2)
ax.plot(g["z"], -g["r"], color=BLUE, lw=2.2)
for s in (1, -1):
    ax.plot([zo, g["z"][-1]], [0, s * g["r"][-1]], color=MUTED, lw=0.9)
    ax.plot([g["z"][-1], zi], [s * g["r"][-1], 0], color=MUTED, lw=0.9)
ax.plot([zo, zi], [0, 0], color=MUTED, lw=0.8, ls=(0, (5, 4)))
ax.plot(zo, 0, "o", color=INK, ms=6); ax.plot(zi, 0, "*", color=RED, ms=13)
ax.annotate("A", (zo, 0), xytext=(0, 9), textcoords="offset points", ha="center")
ax.annotate("A′", (zi, 0), xytext=(0, 9), textcoords="offset points", ha="center")
ax.set_aspect("equal"); ax.set_xlabel("z  [λ]"); ax.set_ylabel("r  [λ]")
ttl(ax, "Cartesian oval separating n₁ = 1.0 from n₂ = 1.5",
    f"A at z₀ = {zo:.0f}λ, A′ at z = {zi:.0f}λ, NA = {NA:.3f}; marginal ray drawn")
plt.show()

## 1. Stigmatism, measured

`opd_waves` rebuilds $d_1$ and $d_2$ from the $(r,z)$ coordinates and
compares $n_1d_1+n_2d_2$ with the axial value $C$.  It never reads the
parametrisation, so a constant result is an observation about the shape.

In [ ]:
opd = opd_waves(itf, g, LAM)
print(f"OPD over the whole aperture:  PV = {np.ptp(opd):.2e} waves,"
      f"  max|OPD| = {np.abs(opd).max():.2e} waves")

## 2. Snell's law at every point

In [ ]:
res = np.abs(n1 * g["sin_i1"] - n2 * g["sin_i2"]).max()
print(f"max |n1 sin i1 − n2 sin i2| = {res:.2e}")
print(f"incidence angle at the rim: i1 = {np.degrees(np.arccos(g['cos_i1'][-1])):.2f}°,"
      f"  i2 = {np.degrees(np.arccos(g['cos_i2'][-1])):.2f}°")

## 3. The ray tubes

A pencil of rays leaving A into $d\Omega_1$ arrives at A′ inside
$d\Omega_2$.  Writing $w=\sin\theta\,d\theta/dd_1$ for the two solid-angle
densities, pure geometry forces

$$\frac{w_2}{w_1}=\frac{\cos i_2}{\cos i_1}\left(\frac{d_1}{d_2}\right)^2 .$$

In [ ]:
lhs = g["w2"] / g["w1"]
rhs = (g["cos_i2"] / g["cos_i1"]) * (g["d1"] / g["d2"]) ** 2
print(f"max relative difference: {np.abs(lhs / rhs - 1).max():.2e}")

## 4. The pupil on the exit sphere

Two derivations, one from the field amplitude and one from the energy
balance in the ray tube:

$$P = t_s\,\frac{d_2}{d_1}
    \qquad\text{and}\qquad
    P = \sqrt{T_s\,\frac{n_1}{n_2}\,\frac{w_1}{w_2}} .$$

$t_s$ is the scalar (TE) Fresnel coefficient — the only one the scalar
boundary conditions $[\psi]=0$, $[\partial_n\psi]=0$ produce.

In [ ]:
P_direct = stigmatic_pupil(itf, g)
P_tubes = stigmatic_pupil_from_tubes(itf, g)
print(f"max relative difference between the two forms: "
      f"{np.abs(P_direct / P_tubes - 1).max():.2e}")
print(f"P(axis) = {P_direct[0]:.4f},  P(rim) = {P_direct[-1]:.4f},"
      f"  edge/axis = {P_direct[-1] / P_direct[0]:.4f}")

In [ ]:
A_ray = predicted_pupil(itf, g)          # absolute: A0 = 1/4pi, never fitted
A_flat = np.full_like(A_ray, A_ray[0])   # the constant-amplitude hypothesis
th2 = np.degrees(g["th2"])

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax[0].plot(th2, A_ray / A_ray[0], color=GREEN, lw=2.4, label="$P = t_s d_2/d_1$")
ax[0].plot(th2, A_flat / A_ray[0], color=MUTED, lw=1.8, ls=(0, (6, 4)),
           label="constant amplitude")
ax[0].plot(th2, t_s(n1, n2, g["cos_i1"], g["cos_i2"]) / t_s(n1, n2, 1.0, 1.0),
           color=BLUE, lw=1.8, ls=(0, (2, 2)), label="$t_s$ alone")
ax[0].set_xlabel(r"$\theta_2$  [deg]"); ax[0].set_ylabel("|A| / |A(0)|")
ax[0].set_ylim(0, 1.1); ax[0].legend(fontsize=9)
ttl(ax[0], "Ray amplitude on the exit sphere",
    "the $d_2/d_1$ factor, not $t_s$, does most of the work")

E_ray = energy_through_sphere(A_ray, g["th2"])
E_flat = energy_through_sphere(A_flat, g["th2"])
ax[1].plot(th2, np.cumsum(np.abs(A_ray) ** 2 * np.sin(g["th2"])) /
           np.sum(np.abs(A_ray) ** 2 * np.sin(g["th2"])), color=GREEN, lw=2.4,
           label="$P = t_s d_2/d_1$")
ax[1].plot(th2, np.cumsum(np.abs(A_flat) ** 2 * np.sin(g["th2"])) /
           np.sum(np.abs(A_flat) ** 2 * np.sin(g["th2"])), color=MUTED, lw=1.8,
           ls=(0, (6, 4)), label="constant amplitude")
ax[1].set_xlabel(r"$\theta_2$  [deg]"); ax[1].set_ylabel("enclosed power / total")
ax[1].legend(fontsize=9)
ttl(ax[1], "Where the power sits on the sphere",
    f"total power ratio  E(flat)/E(ray) = {E_flat / E_ray:.3f}")
plt.tight_layout(); plt.show()

A field of constant amplitude on a sphere carries
`E_flat/E_ray` times the power the refracted field actually carries, and it
is not a solution of the transmission problem: it satisfies neither the
Fresnel amplitude at the surface nor the ray-tube spreading between the
surface and the sphere.  The next notebooks measure the real thing.